# 🐝 A Multi-Agent Swarm: RAG, NL2SQL and Invoice Agents

| Property | Value |
|---|---|
| Origin | LangGraph swarm pattern (`langgraph-swarm`); conceptual ancestors: OpenAI Swarm (2024), AutoGen (Microsoft, 2023) |

In a **swarm** there is no supervisor. Each agent has **handoff tools** it can call to transfer the conversation
to a peer, and the swarm remembers which agent was **last active**, so the next message in the same thread goes
straight back to it. We build three agents over a music-store business:

* **General QA agent (RAG)** — company knowledge from `assets/data/`
* **NL2SQL agent** — catalogue questions answered with SQL over the Chinook sample database
* **Invoice agent** — a customer's invoices and support representative, via fixed, parameterised queries

![](../assets/Multi-Agent-Architecture.png)

## Learning Objectives
In this notebook, you will learn:
1. **Three ReAct sub-agents** - built with `create_agent`, each with its own tools, tested on its own first
2. **Safe database tools** - a read-only SQLite connection for model-written SQL, bound parameters for fixed queries
3. **Handoff tools and `create_swarm`** - let every agent transfer control to every other, and see the `Command` a handoff tool returns
4. **Swarm memory** - a checkpointer keeps the thread and the `active_agent` across turns

## Where this fits
`01_agent_handoffs.ipynb` built handoffs by hand: a tool that returns `Command(goto=…, graph=Command.PARENT)`,
an `active_agent` field, a checkpointer and a `START` router. `langgraph-swarm` packages exactly that:
`create_handoff_tool` builds the tool, and `create_swarm` builds the parent graph and its router. Here every agent can
hand off to every other, and the swarm resumes with whichever agent the last handoff chose.

## Prerequisites
- `01_agent_handoffs.ipynb`; agents built with `create_agent`, and tools
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env` (chat model and embeddings)
- Internet access on the first run: the Chinook SQLite file (about 1 MB) is downloaded once, from a pinned commit, to `chinook.db` beside the notebook (git-ignored)

---
## 📦 Step 1: Environment Setup

The setup cell loads the API keys from the project-root `.env` and silences library deprecation warnings.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: all three agents and the RAG answer chain
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini", temperature=0)

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🗄️ Part 1: The Chinook Sample Database, Read-Only

The [Chinook database](https://github.com/lerocha/chinook-database) holds a music store's customers, invoices,
employees and catalogue. The cell downloads the SQLite file once, from a pinned commit, checks it against the git blob
id of that file, and saves it as `chinook.db` beside the notebook (`*.db` is git-ignored).

The NL2SQL agent will run SQL that a model wrote, so the connection only reads, three times over:
- **`file:chinook.db?mode=ro`** (an SQLite URI) opens the file read-only
- **`PRAGMA query_only = ON`** makes the connection refuse every write, temporary tables included
- **An authorizer** (`conn.set_authorizer`) allows only reads. SQLite calls it for every action while it compiles a
  statement, so `ATTACH DATABASE` (which would open *any* SQLite file on this machine), `PRAGMA query_only = OFF`
  and every write are refused before they run. Read-only flags alone don't stop `ATTACH`: it reads another file
  without writing anything.

The tools are plain Python functions over that connection (`list_tables`, `describe_table`, `run_query`), with no
database wrapper library in between. The cell ends by sending a `DELETE`, an `ATTACH` and a `PRAGMA` through
`run_query` and checking in code that SQLite refused all three.

In [ ]:
# ============================================================================
# PART 1: DOWNLOAD CHINOOK ONCE, OPEN IT READ-ONLY, AND THE SQL TOOLS
# ============================================================================
import hashlib
import sqlite3
from pathlib import Path

import requests
from langchain_core.tools import tool

# Pinned to a commit, so the file never changes under the notebook
CHINOOK_COMMIT = "7f67772503d71ba90f19283c38e93923addb43fa"
CHINOOK_URL = (
    f"https://raw.githubusercontent.com/lerocha/chinook-database/{CHINOOK_COMMIT}"
    "/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite"
)
CHINOOK_BLOB = "f385c2651f80914c3b93c62431b0b000465a921b"  # git blob id of that file at that commit
CHINOOK_DB = Path("chinook.db")  # beside the notebook; *.db is git-ignored
MAX_ROWS = 20  # the most rows one query returns to a model

if not CHINOOK_DB.exists():
    data = requests.get(CHINOOK_URL, timeout=60).content
    # The id `git hash-object` computes: proves this is exactly the pinned file
    blob = hashlib.sha1(b"blob %d\0" % len(data) + data).hexdigest()
    assert blob == CHINOOK_BLOB, f"Download does not match the pinned file (blob {blob}); re-run to retry"
    CHINOOK_DB.write_bytes(data)

# Read-only three times over: the file is opened read-only, the connection refuses writes,
# and the authorizer allows only reads. check_same_thread=False: agents run their tools in worker threads.
conn = sqlite3.connect(f"file:{CHINOOK_DB}?mode=ro", uri=True, check_same_thread=False)
conn.execute("PRAGMA query_only = ON")

# SQLite asks this function about every action while compiling a statement. Only reads pass:
# ATTACH (opening another file), PRAGMA, writes and schema changes are refused before they run.
READ_ACTIONS = {sqlite3.SQLITE_SELECT, sqlite3.SQLITE_READ, sqlite3.SQLITE_FUNCTION, sqlite3.SQLITE_RECURSIVE}


def allow_reads_only(action, arg1, arg2, db_name, trigger) -> int:
    return sqlite3.SQLITE_OK if action in READ_ACTIONS else sqlite3.SQLITE_DENY


conn.set_authorizer(allow_reads_only)


def fetch(sql: str, params=()) -> str:
    """Run ONE statement; return a header line plus at most MAX_ROWS rows, as text."""
    cursor = conn.execute(sql, params)
    rows = cursor.fetchmany(MAX_ROWS)
    if not rows:
        return "No rows."
    header = " | ".join(column[0] for column in cursor.description)
    return "\n".join([header, *(" | ".join(str(value) for value in row) for row in rows)])


@tool
def list_tables() -> str:
    """List the tables in the music store database."""
    tables = conn.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")
    return ", ".join(row[0] for row in tables)


@tool
def describe_table(table: str) -> str:
    """Return the CREATE TABLE statement (columns, types, keys) of one table."""
    row = conn.execute("SELECT sql FROM sqlite_master WHERE type = 'table' AND name = ?", (table,)).fetchone()
    return row[0] if row else f"No table named {table!r}; call list_tables first."


@tool
def run_query(sql: str) -> str:
    """Run ONE read-only SQL query (SQLite dialect) on the music store database; returns up to 20 rows."""
    try:
        return fetch(sql)
    except sqlite3.Error as error:
        return f"Error: {error}"  # the agent reads the error and can fix its query


print("Tables:", list_tables.invoke({}))
print(describe_table.invoke({"table": "Invoice"}))

# Attacks through the same tool the agent uses: no LLM needed to test the guard
before = fetch("SELECT COUNT(*) AS invoices FROM Invoice")
attempts = {
    "a write": "DELETE FROM Invoice",
    "opening another database file": "ATTACH DATABASE ':memory:' AS other",
    "switching the guard off": "PRAGMA query_only = OFF",
}
results = {name: run_query.invoke({"sql": sql}) for name, sql in attempts.items()}
after = fetch("SELECT COUNT(*) AS invoices FROM Invoice")
print()
for name, sql in attempts.items():
    print(f"{sql} -> {results[name]}")
print(f"Invoices before / after: {before.splitlines()[1]} / {after.splitlines()[1]}")
refused = all(result.startswith("Error") for result in results.values()) and before == after
print("✅ write, ATTACH and PRAGMA were all refused; no row changed"
      if refused else "❌ an attempt was not refused")

---
## 🧾 Part 2: Shared State and Thread Memory

`create_swarm` needs one field besides `messages`: **`active_agent`**, the agent that handles the next message.
`langgraph_swarm.SwarmState` is exactly that, `MessagesState` plus `active_agent`: the same shape as `HandoffState` in
`01_agent_handoffs.ipynb`. `create_swarm` narrows `active_agent` to the agents' names. The checkpointer keeps the
messages and `active_agent` per `thread_id`. Agents built with `create_agent` need no extra state field.

In [ ]:
# ============================================================================
# PART 2: SWARM STATE AND CHECKPOINTER
# ============================================================================
from langgraph.checkpoint.memory import InMemorySaver
from langgraph_swarm import SwarmState

checkpointer = InMemorySaver()
print(f"✅ SwarmState fields: {list(SwarmState.__annotations__)}; checkpointer ready")

---
## 📚 Part 3: Sub-Agent 1 — General QA over the Knowledge Base (RAG)

The knowledge base is the text files in `assets/data/`, read into `Document`s, split into chunks and embedded into an
in-memory Chroma collection. The chunk ids are fixed (`chunk-0`, `chunk-1`, …), so re-running the cell replaces the
chunks instead of adding copies; the cell prints the collection size to show it.

Every agent in this notebook is built with `langchain.agents.create_agent` (`system_prompt=` sets its role), the
constructor `langgraph-swarm`'s own README and docstrings use.

In [ ]:
# ============================================================================
# PART 3A: THE RAG TOOL
# ============================================================================
from aied_helpers import get_embeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

KNOWLEDGE_DIR = Path("../assets/data")

# One Document per text file: its text, plus where it came from
documents = [
    Document(page_content=path.read_text(encoding="utf-8"), metadata={"source": str(path)})
    for path in sorted(KNOWLEDGE_DIR.glob("*.txt"))
]
splits = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(documents)
vectorstore = Chroma.from_documents(splits, get_embeddings(provider="openai", model="text-embedding-3-small"),
                                    ids=[f"chunk-{i}" for i in range(len(splits))],  # fixed ids: re-runs upsert
                                    collection_name="swarm_knowledge")
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print(f"Indexed {len(documents)} document(s) as {len(splits)} chunks; "
      f"the collection holds {len(vectorstore.get()['ids'])}")

rag_chain = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using only the context. If the context does not contain the answer, say so."),
    ("human", "Question: {question}\n\nContext:\n{context}"),
]) | llm | StrOutputParser()


@tool
def rag_tool(query: str) -> str:
    """Answer general questions about the company (policies, services, locations) from the knowledge base."""
    context = "\n\n".join(doc.page_content for doc in retriever.invoke(query))
    return rag_chain.invoke({"question": query, "context": context})

In [ ]:
# ============================================================================
# PART 3B: THE GENERAL QA AGENT, TESTED ON ITS OWN
# ============================================================================
import uuid

from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, ToolMessage

general_qa_subagent_prompt = """You are a subagent among a team of assistants, specialised in general questions about the
company, its policies and services, answered from the knowledge base with `rag_tool`.

IMPORTANT RULES:
- Do NOT answer questions about music albums, artists, tracks, customers, employees or invoices — those live in the
  database and belong to the NL2SQL or invoice agents. Transfer the user to the right agent instead.
- Ground every answer in the retrieved context; if it is missing, say so.
"""

general_qa_subagent = create_agent(model=llm, tools=[rag_tool], system_prompt=general_qa_subagent_prompt,
                                   name="general_qa_agent", checkpointer=checkpointer)


def show(result) -> None:
    for message in result["messages"]:
        message.pretty_print()


show(general_qa_subagent.invoke({"messages": [HumanMessage(content="Tell me about the customer service policies.")]},
                                config={"configurable": {"thread_id": str(uuid.uuid4())}}))

---
## 🧮 Part 4: Sub-Agent 2 — NL2SQL over the Catalogue

The agent writes the SQL itself. `list_tables` and `describe_table` show it the schema; `run_query` runs its query
and returns the rows or the error, so it can fix a bad query and try again. Model-written SQL is untrusted, and the
guard does not depend on the model: the connection from Part 1 only reads (read-only file, `query_only`, and an
authorizer that refuses `ATTACH`, `PRAGMA` and writes), so SQLite rejects anything else whatever the SQL says.

In [ ]:
# ============================================================================
# PART 4: THE NL2SQL AGENT, TESTED ON ITS OWN
# ============================================================================
sql_tools = [list_tables, describe_table, run_query]

nl2sql_subagent_prompt = """You are a subagent among a team of assistants, specialised in questions answered from the
database: albums, artists, tracks, genres and other catalogue data. Use `list_tables` and `describe_table` to find the
tables and columns you need, then answer with `run_query` (one SELECT at a time; fix the query if it returns an error).

IMPORTANT RULES:
- Handle all catalogue questions yourself; do not delegate them to the general QA agent.
- Only transfer to general QA for company policies or services, and to the invoice agent for a customer's invoices.
- Present results in plain language, not raw SQL.
"""

nl2sql_subagent = create_agent(model=llm, tools=sql_tools, system_prompt=nl2sql_subagent_prompt,
                               name="nl2sql_agent", checkpointer=checkpointer)

result = nl2sql_subagent.invoke({"messages": [HumanMessage(content="Do you have any albums by the Rolling Stones?")]},
                                config={"configurable": {"thread_id": str(uuid.uuid4())}})
show(result)
ran_sql = any(isinstance(m, ToolMessage) and m.name == "run_query" for m in result["messages"])
print("✅ the agent answered from a query it wrote and ran" if ran_sql else "❌ the agent never called run_query")

---
## 🧾 Part 5: Sub-Agent 3 — Invoices

The invoice tools run fixed queries with the customer ID as a **bound parameter** (`:customer_id`), never pasted
into the SQL text — the model supplies the value, so it must not be able to change the query.

In [ ]:
# ============================================================================
# PART 5A: PARAMETERISED INVOICE TOOLS
# ============================================================================
@tool
def get_invoices_by_customer_sorted_by_date(customer_id: int) -> str:
    """Look up all invoices for a customer by customer ID, most recent first."""
    return fetch("SELECT * FROM Invoice WHERE CustomerId = :customer_id ORDER BY InvoiceDate DESC",
                 {"customer_id": customer_id})


@tool
def get_invoices_sorted_by_unit_price(customer_id: int) -> str:
    """Look up a customer's invoice lines sorted by unit price, highest first."""
    return fetch(
        "SELECT Invoice.*, InvoiceLine.UnitPrice FROM Invoice "
        "JOIN InvoiceLine ON Invoice.InvoiceId = InvoiceLine.InvoiceId "
        "WHERE Invoice.CustomerId = :customer_id ORDER BY InvoiceLine.UnitPrice DESC",
        {"customer_id": customer_id},
    )


@tool
def get_employee_by_invoice_and_customer(invoice_id: int, customer_id: int) -> str:
    """Return the support employee associated with a customer's invoice."""
    info = fetch(
        "SELECT Employee.FirstName, Employee.Title, Employee.Email FROM Employee "
        "JOIN Customer ON Customer.SupportRepId = Employee.EmployeeId "
        "JOIN Invoice ON Invoice.CustomerId = Customer.CustomerId "
        "WHERE Invoice.InvoiceId = :invoice_id AND Invoice.CustomerId = :customer_id",
        {"invoice_id": invoice_id, "customer_id": customer_id},
    )
    return info if info != "No rows." else f"No employee found for invoice {invoice_id} and customer {customer_id}."


invoice_tools = [get_invoices_by_customer_sorted_by_date, get_invoices_sorted_by_unit_price,
                 get_employee_by_invoice_and_customer]
print("✅ Invoice tools:", [t.name for t in invoice_tools])

In [ ]:
# ============================================================================
# PART 5B: THE INVOICE AGENT, WITH AND WITHOUT A CUSTOMER ID
# ============================================================================
invoice_subagent_prompt = """You are a subagent among a team of assistants, specialised in a customer's invoices.
Use your tools to find invoices (by date or by unit price) and the support employee for an invoice.
You need the customer's ID: if the user has not given it, ask for it. If you cannot retrieve the information,
say so and ask whether they want to search for something else.
"""

invoice_information_subagent = create_agent(model=llm, tools=invoice_tools, system_prompt=invoice_subagent_prompt,
                                            name="invoice_information_agent", checkpointer=checkpointer)

for question, needs_tools in [
    ("What was my most recent invoice, and who was the employee that helped me with it?", False),
    ("My customer id is 2. What was my most recent invoice, and who was the employee that helped me with it?", True),
]:
    print(f"\n################ {question}")
    result = invoice_information_subagent.invoke({"messages": [HumanMessage(content=question)]},
                                                 config={"configurable": {"thread_id": str(uuid.uuid4())}})
    show(result)
    used_tools = any(isinstance(m, ToolMessage) for m in result["messages"])
    print(("✅" if used_tools == needs_tools else "❌")
          + (" called the invoice tools" if used_tools else " asked for the customer ID instead of calling a tool"))

Without an ID the agent should ask for one (it has no other way to know who you are); with "customer id is 2" it
should call the tools and answer with the latest invoice and the support representative. The ✅ line under each run
checks which of the two happened.

---
## 🐝 Part 6: The Swarm

Each agent gets its own tools **plus handoff tools** to the other two (`create_handoff_tool`). `create_swarm` builds
the parent graph that `01_agent_handoffs.ipynb` built by hand: one node per agent and a `START` router on
`active_agent`, with `general_qa_agent_with_handoff` as the default first agent. Compiling with the checkpointer makes
the swarm remember the active agent per thread.

In [ ]:
# ============================================================================
# PART 6A: AGENTS WITH HANDOFF TOOLS, AND THE SWARM
# ============================================================================
from langgraph_swarm import create_handoff_tool, create_swarm

GENERAL = "general_qa_agent_with_handoff"
NL2SQL = "nl2sql_agent_with_handoff"
INVOICE = "invoice_information_agent_with_handoff"

to_general = create_handoff_tool(
    agent_name=GENERAL, description="Transfer the user to the general QA agent for company, policy or service questions")
to_nl2sql = create_handoff_tool(
    agent_name=NL2SQL, description="Transfer the user to the NL2SQL agent for music catalogue questions")
to_invoice = create_handoff_tool(
    agent_name=INVOICE, description="Transfer the user to the invoice agent for questions about their invoices")

general_qa_agent_with_handoff = create_agent(
    model=llm, tools=[rag_tool, to_invoice, to_nl2sql], system_prompt=general_qa_subagent_prompt, name=GENERAL)
nl2sql_agent_with_handoff = create_agent(
    model=llm, tools=[*sql_tools, to_general, to_invoice], system_prompt=nl2sql_subagent_prompt, name=NL2SQL)
invoice_information_agent_with_handoff = create_agent(
    model=llm, tools=[*invoice_tools, to_general, to_nl2sql], system_prompt=invoice_subagent_prompt, name=INVOICE)

swarm_agents = create_swarm(
    agents=[general_qa_agent_with_handoff, nl2sql_agent_with_handoff, invoice_information_agent_with_handoff],
    default_active_agent=GENERAL,
    state_schema=SwarmState,
).compile(checkpointer=checkpointer)

for edge in swarm_agents.get_graph().edges:
    print(f"  {edge.source} -> {edge.target}")

### 6.1 🔍 Inside a `langgraph-swarm` Handoff Tool

`create_handoff_tool` builds the same thing `01_agent_handoffs.ipynb` built by hand. Call one directly, with a full
tool call and a stand-in for the injected state (no model), and look at what it returns.

In [ ]:
# ============================================================================
# PART 6B: THE COMMAND A LIBRARY HANDOFF TOOL RETURNS (no LLM)
# ============================================================================
from langgraph.types import Command

command = to_invoice.invoke({
    "type": "tool_call",  # a full tool call, so the tool receives "call_demo" as its call id
    "id": "call_demo",
    "name": to_invoice.name,
    "args": {"state": {"messages": [HumanMessage(content="What was my most recent invoice?")]}},
})
print(f"tool name:    {to_invoice.name}")
print(f"goto:         {command.goto}")
print(f"graph:        {command.graph}")
print(f"update keys:  {list(command.update)}")
print(f"last message: {command.update['messages'][-1].content}")

same_shape = (
    command.goto == INVOICE
    and command.graph == Command.PARENT
    and set(command.update) == {"messages", "active_agent"}
    and command.update["active_agent"] == INVOICE
)
print("✅ the same Command as the hand-built tool in 01_agent_handoffs.ipynb: run the agent, in the parent graph, "
      "with the messages and active_agent" if same_shape else "❌ unexpected Command")

The one difference from the hand-built tool: no `context` argument. The library's tool takes none, so whatever the
next agent needs must already be in the conversation, and its `ToolMessage` only says "Successfully transferred".

In [ ]:
# ============================================================================
# PART 6C: THREE QUESTIONS, THREE DESTINATIONS
# ============================================================================
from langchain_core.messages import AIMessage


def ask_swarm(question: str, thread_id: str, expected: list[str]) -> bool:
    """Send one message on a thread; print which agents acted this turn and check them against `expected`."""
    config = {"configurable": {"thread_id": thread_id}}
    result = swarm_agents.invoke({"messages": [HumanMessage(content=question)]}, config=config)

    # Only this turn: the AIMessages after the last user message, by agent name, repeats collapsed
    last_human = max(i for i, m in enumerate(result["messages"]) if isinstance(m, HumanMessage))
    agents = []
    for m in result["messages"][last_human + 1 :]:
        if isinstance(m, AIMessage) and m.name and (not agents or agents[-1] != m.name):
            agents.append(m.name)

    ok = agents == expected
    print(f"Q: {question}")
    print(f"   agents this turn:  {' → '.join(agents) or '(none)'}")
    print(f"   active agent now:  {swarm_agents.get_state(config).values.get('active_agent')}")
    print(f"   answer: {result['messages'][-1].content[:300]}")
    print(f"   {'✅' if ok else '❌'} expected {' → '.join(expected)}\n")
    return ok


checks = [
    ask_swarm("Do you have any albums by the Rolling Stones?", str(uuid.uuid4()), [GENERAL, NL2SQL]),
    ask_swarm("My customer id is 1. What was my most recent invoice, and who was the employee that helped me with it?",
              str(uuid.uuid4()), [GENERAL, INVOICE]),
    ask_swarm("Tell me about the company location.", str(uuid.uuid4()), [GENERAL]),
]
print(f"{sum(checks)}/{len(checks)} questions reached the expected agent")

Every question starts at the general QA agent: a new thread has no `active_agent`, so the router uses the default.
The catalogue and invoice questions should show a handoff in "agents this turn", and "active agent now" names the agent
the last handoff chose. For the location question no handoff happens, so it stays `None`.

---
## 🔁 Part 7: Multi-Turn — the Swarm Remembers Who Was Active

In one thread, after a handoff the next message goes **straight to the last active agent** — no new routing.
The scripted conversation below replaces an interactive `input()` loop so the notebook runs unattended.

In [ ]:
# ============================================================================
# PART 7: ONE THREAD, THREE TURNS
# ============================================================================
thread = str(uuid.uuid4())
checks = [
    ask_swarm("My customer id is 3. What was my most recent invoice?", thread, [GENERAL, INVOICE]),
    ask_swarm("And which employee helped me with it?", thread, [INVOICE]),  # should stay with the invoice agent
    ask_swarm("What are your customer service hours?", thread, [INVOICE, GENERAL]),  # should hand off to general QA
]
print(f"{sum(checks)}/{len(checks)} turns took the expected path")

After turn 1 the active agent is the invoice agent, so turn 2, which only makes sense with turn 1's context, is
answered by it directly. Turn 3 is a policy question, so the invoice agent hands off to general QA. The ✅ lines check
each turn's path.

---
## 📝 Summary

In this notebook, we learned:

### 1. Three ReAct sub-agents
- RAG over the knowledge base, NL2SQL over the catalogue, and fixed invoice queries, each built with `create_agent` and tested alone first

### 2. Safe database tools
- Model-written SQL runs on a connection that only reads (`mode=ro`, `PRAGMA query_only` and a read-only authorizer); a write, an `ATTACH` and a `PRAGMA` are rejected, checked in code
- Fixed queries bind parameters instead of pasting values; the Chinook file comes from a pinned commit and is checked against its git blob id

### 3. Handoff tools and `create_swarm`
- Every agent can transfer control to every other; there is no supervisor
- A library handoff tool returns the same `Command(goto=…, graph=Command.PARENT, update={…, "active_agent": …})` as the hand-built one in `01_agent_handoffs.ipynb`

### 4. Swarm memory
- The checkpointer keeps `active_agent` per thread, so follow-ups go straight back to the agent that was helping

### Next Steps
- **Module 08 — `08_collaboration_and_control_patterns`**: blackboards, debate, STORM, meta-controllers and other ways agents collaborate